# Soial Media Sentiment Analysis

In [ ]:
%pip install tensorflow

In [ ]:
import tensorflow as tf
tf.__version__

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
# from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import re
import string

# Data Preview

In [ ]:
sentiment_data = pd.read_csv('/kaggle/input/sentimentdataset-csv/sentimentdataset.csv')

In [ ]:
sentiment_data.info()

732 entries, not a large dataset

Notice two columns are missing appropirate labels, need to label all columns with meanings and drop duplicate ones.

In [ ]:
# rename columns 'unamed' to 'id'
sentiment_data.rename(columns={'Unnamed: 0': 'id'}, inplace=True)

In [ ]:
# drop remaining column 'unamed: 0.1'
sentiment_data.drop(columns=['Unnamed: 0.1'], inplace=True)

In [ ]:
# reset index as column 'id'
sentiment_data.set_index('id', inplace=True)

In [ ]:
sentiment_data.head()

In [ ]:
# check for missing values
sentiment_data.isnull().sum()

NULL check: no NULL value

In [ ]:
# check for duplicates
sentiment_data.duplicated().sum()

Duplicate check: 20 duplicate values, need to eliminate

In [ ]:
sentiment_data.drop_duplicates(inplace=True)

In [ ]:
sentiment_data.duplicated().sum()

In [ ]:
sentiment_data.info()

### Question 1: Are sentiments aligned with the original post?

# Sentiment Text Analysis using Neural Network

Now we want to know whether each post is labeled correctly by analyzing the original text and comparing the predicted output with the actual output using **neural network**. Given a model accuracy >= 0.9, the optimized model can be generated to re-align texts and sentiments in the original daataset.

This is a multi-class classification problem, and we will use Long Short-term Model for text classification.

### sentiment data preprocessing

In [ ]:
# counts the top 10 most frequent words in the 'Sentiment' column
sentiment_data['Sentiment'].value_counts()

Top sentiments include: positive, joy, excitement....

In [ ]:
# Visualize the top 10 most frequent words in the 'Sentiment' column, and order by frequency in descending order
sentiment_data['Sentiment'].value_counts().head(10).plot(kind='bar')
plt.title('Top 10 sentiments on social media sites')

In [ ]:
# list out all sentiments
pd.set_option('display.max_rows', None)
sentiment_data['Sentiment'].value_counts()

Noticed some sentiments are duplicated, need to trim data for accuracy

In [ ]:
# convert all texts to lowercase & strip the white spaces in the 'Sentiment' column
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].str.lower()
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].str.strip()
sentiment_data['Sentiment'].value_counts()

Many sentiments are listed here...but can some of them grouped together?

### Synonym Replacement

In [ ]:
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Positive': 'Happy', 'Joy' : 'Happy', 'Serenity' : 'Happy', 'Euphoria' : 'Happy', 'Elation' : 'Happy', 'Happiness' : 'Happy', 'Playful' : 'Happy', 'Amusement' : 'Happy'})
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Despair' : 'Sad', 'Grief' : 'Sad', 'Regret' : 'Sad', 'Melancholy' : 'Sad', 'Negative' : 'Sad', 'Bad' : 'Sad', 'Loneliness' : 'Sad', 'Desolation' : 'Sad'})
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Excitement' : 'Excited', 'Thrill' : 'Excited', 'Adventure' : 'Excited', 'Enthusiasm' : 'Excited', 'Inspired' : 'Excited', 'Inspiration' : 'Excited', 'Arousal' : 'Excited'})
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Hate' : 'Angry', 'Disgust' : 'Angry', 'Bitterness' : 'Angry', 'Betrayal' : 'Angry', 'Frustration' : 'Angry', 'Frustrated' : 'Angry', 'Anger' : 'Angry'})
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Pride' : 'Proud', 'Admiration' : 'Proud', 'Awe' : 'Proud', 'Reverence' : 'Proud'})
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Contentment' : 'Content', 'Acceptance' : 'Content', 'Serenity' : 'Content', 'Fulfillment' : 'Content', 'Calmness' : 'Content', 'Satisfaction' : 'Content'})
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Indifference' : 'Neutral', 'Numbness' : 'Neutral', 'Indifference' : 'Neutral', 'Ambivalence' : 'Neutral'})
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Hope' : 'Hopeful', 'Determination' : 'Hopeful', 'Resilience' : 'Hopeful', 'Empowerment' : 'Hopeful'})
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Shame' : 'Embarassed', 'Embarassment' : 'Embarassed'})
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Gratitude' : 'Grateful'})
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Compassionate' : 'Compassion', 'Tenderness' : 'Compassion', 'Empathetic' : 'Compassion'})

In [ ]:
sentiment_data['Sentiment'].value_counts()

keep grouping more sentiments until the majority of sentiments are clustered.

In [ ]:
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Compassion' : 'Compassionate'})
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Nostalgia' : 'Nostalgic'})
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Confusion' : 'Confused'})
sentiment_data['Sentiment'] = sentiment_data['Sentiment'].replace({'Surprise' : 'Surprised'})
sentiment_data['Sentiment'].value_counts()

In [ ]:
sentiment_data['Sentiment'].value_counts().nlargest(12).plot(kind='bar')

In [ ]:
# make a copy of the original df for NLP
sentiment_text = sentiment_data.copy()
sentiment_text = sentiment_text[['Text', 'Sentiment']]
sentiment_text.head()

In [ ]:
sentiment_text['Sentiment'].value_counts()

We will start with a small sample of the overall dataset for training purpose - corpus will include texts beloinging to major sentiment groups for better training quality.

In [ ]:
## trimming the text dataset to only include the major sentiments (top 20) for better clustering
top_20_sentiments = sentiment_text['Sentiment'].value_counts().nlargest(20).index
sentiment_text = sentiment_text[sentiment_text['Sentiment'].isin(top_20_sentiments)]

## Label encoding

we need to assign each sentiment to a unique identifier before analyzing them with neural network

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
# label encoding different sentiment groups
le = LabelEncoder()
sentiment_text['Sentiment'] = le.fit_transform(sentiment_text['Sentiment'])
sentiment_text.head()

In [ ]:
sentiment_text['Sentiment'].value_counts()

In [ ]:
sentiment_text.info()

## Input & Output Preprocessing

In [ ]:
# prepare input & output
X = sentiment_text['Text'].to_numpy().ravel() # Prepare sentences for input
X = tf.strings.lower(X)
X = tf.strings.regex_replace(X,
                                  '[%s]' % re.escape(string.punctuation), # leave special characters in
                                  '') # strip all punctuations
y = sentiment_text['Sentiment'].to_numpy().ravel()

In [ ]:
print(X)

In [ ]:
print(y)

In [ ]:
X.shape
y.shape

In [ ]:
# convert text & sentiments to tensors
X = tf.convert_to_tensor(X)
y = tf.convert_to_tensor(y)

In [ ]:
# create random indices to shuffle the dataset
indices = tf.range(start=0, limit=tf.shape(X)[0], dtype=tf.int32)
shuffled_indices = tf.random.shuffle(indices)

# make sure X & y remain aligned
X = tf.gather(X, shuffled_indices)
y = tf.gather(y, shuffled_indices)

In [ ]:
# separate dataset into training and test

In [ ]:
train_size = int(tf.floor(0.7*len(X)))
X_train = X[:train_size]
y_train = y[:train_size]
X_test = X[train_size:]
y_test = y[train_size:]

In [ ]:
print(len(X_train))
print(len(y_train))
print(len(X_test))
print(len(y_test))

In [ ]:
# make labels into vectors using one-hot encoding
y_train_oh = tf.one_hot(y[:train_size], depth=20)
y_test_oh = tf.one_hot(y[train_size:], depth=20)

Now we have generated datasets, we can proceed to build the model

## Word Vectorization

**Recurrent neural networks (RNNs)** is a preferred neural network designed for processing sequential data (where the order matters) such as text, but traditional RNNs suffer from the vanishing gradient problem.
So we will use **LSTM model** to mitigate this issue.

Noted 

In [ ]:
# set the parameters of the vectorization layer
vocab_size = 50000 # max token in each training 
sequence_length = 20 # input dimension = 20

In [ ]:
word_to_vec = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=sequence_length)

In [ ]:
word_to_vec.adapt(X_train)
X_train = word_to_vec(X_train)
X_test = word_to_vec(X_test)

In [ ]:
# convert input to numpy array 
X_train = np.array(X_train)
X_test = np.array(X_test)

In [ ]:
# make sure input are arrays of int
print(type(X_train[0][0])) 

## Model Building

In [ ]:
model = tf.keras.Sequential()
model.add(tf.keras.layers.LSTM(units=128, input_shape=(20,1), return_sequences=True)) # first LSTM layer, 128 memory cells, return full sequence of hidden states
model.add(tf.keras.layers.Dropout(0.3)) # set 30% neurons to 0
model.add(tf.keras.layers.BatchNormalization())  # add batch Normalization
model.add(tf.keras.layers.LSTM(units=128, input_shape=(20,), return_sequences=False)) # another LSTM layer,output only hidden states
model.add(tf.keras.layers.Dropout(0.3)) # set 30% neurons to 0
model.add(tf.keras.layers.Dense(20, activation='softmax',kernel_regularizer=tf.keras.regularizers.L2(0.01)) ) # output layer, condense to 10 neurons to fit the output


In [ ]:
# optimize model by fine tuning
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005) # reduce learning rate
model.compile(loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False), 
              optimizer=optimizer,
              metrics=['accuracy'])

In [ ]:
# introduce early stopping to prevent overfitting
callback = tf.keras.callbacks.EarlyStopping(monitor='loss',patience=3)

In [ ]:
history = model.fit(
    x=X_train, y=y_train_oh    
    ,epochs=50
    ,batch_size = 5
    # ,callbacks=[callback]
    ,validation_data=(X_test, y_test_oh)
) 

In [ ]:
model.summary()

In [ ]:
# make predictions using test data
predictions = model.predict(X_test)

predicted_classes = predictions.argmax(axis=-1) # get the index of the highest probability

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, predicted_classes))

Given an almost perfect score in training and a 0.28 accuracy score in test data, this is clearly an issue of overfitting.

In [ ]:
# model1 - initial model, overfitting
# model = tf.keras.Sequential()
# model.add(tf.keras.layers.LSTM(units=128, input_shape=(20,1), return_sequences=True)) # first LSTM layer, 128 memory cells, return full sequence of hidden states
# model.add(tf.keras.layers.Dropout(0.3)) # set 30% neurons to 0
# model.add(tf.keras.layers.LSTM(units=128, input_shape=(20,), return_sequences=False)) # another LSTM layer,output only hidden states
# model.add(tf.keras.layers.Dropout(0.3)) # set 30% neurons to 0
# model.add(tf.keras.layers.Dense(10, activation='softmax')) # output layer, condense to 10 neurons to fit the output
#  # use softmax for one vs all classification

# # optimize model by fine tuning
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.003)
# model.compile(loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
#               optimizer=optimizer,
#               metrics=['accuracy'])

# result = model.fit(
#     x=X_train, y=y_train_oh,    
#     epochs=100) 

# Given a 0.98 accuracy score in training and a 0.28 accuracy score in test data, this is clearly an issue of overfitting.

In [ ]:
# model 2 - add batch normalization during model building & train with batch size = 5, still overfitting
# model = tf.keras.Sequential()
# model.add(tf.keras.layers.LSTM(units=128, input_shape=(20,1), return_sequences=True)) # first LSTM layer, 128 memory cells, return full sequence of hidden states
# model.add(tf.keras.layers.Dropout(0.3)) # set 30% neurons to 0
# model.add(tf.keras.layers.BatchNormalization())  # add batch Normalization
# model.add(tf.keras.layers.LSTM(units=128, input_shape=(20,), return_sequences=False)) # another LSTM layer,output only hidden states
# model.add(tf.keras.layers.Dropout(0.3)) # set 30% neurons to 0
# model.add(tf.keras.layers.Dense(10, activation='softmax')) # output layer, condense to 10 neurons to fit the output
#  # use softmax for one vs all classification

# # optimize model by fine tuning
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.003)
# model.compile(loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
#               optimizer=optimizer,
#               metrics=['accuracy'])

# result = model.fit(
#     x=X_train, y=y_train_oh,    
#     epochs=100, batch_size=5) 

# Given a 0.89 accuracy score in training and a 0.30 accuracy score in test data, this is clearly an issue of overfitting.

In [ ]:
# model3 - introduce early stopping to prevent overfitting, much less accuracy in training data but no overfitting
# model = tf.keras.Sequential()
# model.add(tf.keras.layers.LSTM(units=128, input_shape=(20,1), return_sequences=True)) # first LSTM layer, 128 memory cells, return full sequence of hidden states
# model.add(tf.keras.layers.Dropout(0.3)) # set 30% neurons to 0
# model.add(tf.keras.layers.BatchNormalization())  # add batch Normalization
# model.add(tf.keras.layers.LSTM(units=128, input_shape=(20,), return_sequences=False)) # another LSTM layer,output only hidden states
# model.add(tf.keras.layers.Dropout(0.3)) # set 30% neurons to 0
# model.add(tf.keras.layers.Dense(10, activation='softmax')) # output layer, condense to 10 neurons to fit the output
#  # use softmax for one vs all classification

# introduce early stopping to prevent overfitting
# callback = tf.keras.callbacks.EarlyStopping(monitor='loss',patience=3)

# ...
# Given a 0.40 accuracy score in training and a 0.34 accuracy score in test data, more tuning is required.

In [ ]:
# change the activation function other than softmax largely decrease the accuracy, not considered

In [ ]:
# model4 - reduce learning rate & change the loss function to mean squared error
# ...
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005) # reduce learning rate
# model.compile(loss='mean_squared_error', # change loss function
#               optimizer=optimizer,
#               metrics=['accuracy'])
# ...
# Given a 0.91 accuracy score in training and a 0.32 accuracy score in test data, more tuning is required.

In [ ]:
# model5 - add l2 regularization during model building, overfitting and overall less accuracy
# model = tf.keras.Sequential()
# model.add(tf.keras.layers.LSTM(units=128, input_shape=(20,1), return_sequences=True)) # first LSTM layer, 128 memory cells, return full sequence of hidden states
# model.add(tf.keras.layers.Dropout(0.3)) # set 30% neurons to 0
# model.add(tf.keras.layers.BatchNormalization())  # add batch Normalization
# model.add(tf.keras.layers.LSTM(units=128, input_shape=(20,), return_sequences=False)) # another LSTM layer,output only hidden states
# model.add(tf.keras.layers.Dropout(0.3)) # set 30% neurons to 0
# model.add(tf.keras.layers.Dense(10, activation='softmax', kernel_regularizer=tf.keras.regularizers.L2(0.01))) # output layer, condense to 10 neurons to fit the output
# # use softmax as activation for mutlti-class classfication, add L2 regularization to punish large weights

# Given a 0.59 accuracy score in training and a 0.32 accuracy score in test data, more tuning is required.

In [ ]:
# try increasing the size of the training data, still very poor performance

# Exploratory Data Analysis

### Question 2: How did sentiments change over time?

In [ ]:
# how recent is this dataset?
# check the value of 'Year' column
sentiment_data['Year'].value_counts()

Year range from 2010 to 2023, with the majority of posts (287/712) are posted in 2023.

In [ ]:
# draw several plots to visualize top 5 sentiment in each year, along with their numbers of retweets and likes.

# step 1: sort out years as an array for filtering data
years = sentiment_data['Year'].unique()
years.sort()

# step 2: use for loop to draw sentiment distribution for each year in chronological order
for year in years:
    # filter out top 5 sentiments in each year using value_counts
    top_5_sentiment = sentiment_data[sentiment_data['Year'] == year]['Sentiment'].value_counts().head(5)
    # design the basic layout
    fig, ax = plt.subplots(1, 6, figsize=(20, 5))
    # draw pie chart for top 5 sentiment distribution
    top_5_sentiment.plot(kind='pie', ax=ax[0])
    ax[0].set_title(f'Top 5 Sentiment in {year}')
    ax[0].set_ylabel('')   
    # group by each sentiment and calculate the average of retweets and likes
    for i, sentiment in enumerate(top_5_sentiment.index):
        sentiment_group = sentiment_data[(sentiment_data['Year'] == year) & (sentiment_data['Sentiment'] == sentiment)].groupby('Sentiment')[['Retweets', 'Likes']].mean().plot(kind='bar', ax=ax[i+1])
        ax[i+1].set_title(f'{sentiment} in {year}')
        ax[i+1].set_xlabel('')
        ax[i+1].set_ylabel('Average per post')


According to these graphs, I notice two trends:
1. In the 2010-2015, positive sentiments dictate social media posts, up until 2016.
2. As time progresses, sentiments are becoming more diversifed, and negative sentiments start to gain popularity since 2016,
 <br>(This observation can be flawed due to the time imbalance in this dataset)
3. The average number of likes is greater than the average number of retweets, and both figues stay stable over the years.

Now zoom in on the top 5 sentiments in 2023, and see the average of retweets and likes for each sentiment.

In [ ]:
# filter data that only in year 2023
sentiment_data_2023 = sentiment_data[sentiment_data['Year'] == 2023]
sentiment_data_2023.info()

In [ ]:
# use value_counts to filter out top 5 sentiments in 2023
top_5_sentiment_2023 = sentiment_data[sentiment_data['Year'] == 2023]['Sentiment'].value_counts().head(5)
sentiment_data_top_5_2023 = sentiment_data_2023[sentiment_data_2023['Sentiment'].isin(top_5_sentiment_2023.index)]
sentiment_data_top_5_2023.info()

Posts identified with major 5 sentiments in 2023 accounts for almost 46% (131/287) posts in 2023

In [ ]:
# draw pie chart for top 5 sentiment distribution in 2023
top_5_sentiment_2023.plot(kind='pie', autopct='%.2f%%')
plt.ylabel('')
plt.title('Top 5 Sentiment in 2023')

In [ ]:
# draw countplot for the average number of retweets and likes for each sentiment in 2023
# groupby 'Sentiment' and calculate the average number of 'Retweets' and 'Likes'
sentiment_data_top_5_2023.groupby('Sentiment')[['Retweets', 'Likes']].mean().plot(kind='bar')
plt.title('Average number of Retweets and Likes for each sentiment in 2023')
plt.xlabel('Sentiment')
plt.ylabel('Average number per post')

plt.savefig('average_retweets_likes_2023.png')

### Question 3: How about sentiments in different social media platforms? 
<br>Is this dataset representative of sentiments across major platforms?

In [ ]:
# find the most popular platforms, and make sure each platform is only counted once
sentiment_data['Platform'].value_counts()

In [ ]:
# convert all texts to lowercase & strip the white spaces in the 'Platform' column
sentiment_data['Platform'] = sentiment_data['Platform'].str.lower()
sentiment_data['Platform'] = sentiment_data['Platform'].str.strip()
sentiment_data['Platform'].value_counts()

In [ ]:
# visualize the most popular platforms using pie chart
sentiment_data['Platform'].value_counts().plot(kind='pie', autopct='%.2f%%')
plt.title('Most popular platforms')
plt.ylabel('')

It seems out dataset are balanced in terms of sampling from three major social media sites.
<br><br>But is it the case when we consider this composition in different years?

In [ ]:
# use for loop to find major social media platforms for each year
for year in years:
    sentiment_data[sentiment_data['Year'] == year]['Platform'].value_counts().plot(kind='pie', autopct='%.2f%%')
    plt.title(f'Major socia media platforms in {year}')
    plt.ylabel('')
    plt.show()

Noticed Instagram did not emerge until 2011, <br>Despite the overall imbalance in time, 
 this dataset is balanced in sampling from three major social media sites in each year.

### Question 4: How does sentiment differ in different countries?

In [ ]:
# find out what countries are contained in the dataset
sentiment_data['Country'].value_counts()

USA is counted twice, need to trim for accuracy

In [ ]:
# convert all texts to lowercase & strip the white spaces in the 'Platform' column
sentiment_data['Country'] = sentiment_data['Country'].str.lower()
sentiment_data['Country'] = sentiment_data['Country'].str.strip()
sentiment_data['Country'].value_counts()

In [ ]:
# visualize the result in pie chart
plt.figure(figsize=(15, 10))
sentiment_data['Country'].value_counts().plot(kind='pie', autopct='%.2f%%')
plt.legend(loc='best')

Base on the graph, we can see most(73.87%) posts sampled from this dataset come from English-speaking countries (USA, UK, Canasda, Australia).
<br><br>As a foreigner, I personally tend to express positivity when I speak a foreign language, and since all posts are written in English, will nativity of English affect the sentiment distribution?

In [ ]:
# # group countries into 'English' and 'Non-English', and find out their sentiment distribution
# # create a list of English-speaking countries
# english_countries = ['usa', 'uk', 'canada', 'australia', 'india', 'south africa', 'ireland', 'scotland']
# # create a new column 'English' to indicate whather this post come from a Native English speaking country
# sentiment_data['Native Language'] = sentiment_data['Country'].apply(lambda x: 'English' if x in english_countries else 'Non-English')
# # sentiment_data['Native Language'].value_counts()
# # draw pie charts to compare sentiment distribution in English country and non-English country posts
# for language in sentiment_data['Native Language'].unique():
#     sentiment_data[sentiment_data['Native Language'] == language]['Sentiment_Nature'].value_counts().plot(kind='pie', autopct='%.2f%%')
#     plt.title(f'Sentiment distribution in {language} speaking countries')
#     plt.ylabel('')
#     plt.legend(loc='best')
#     plt.show()
#     plt.savefig(f'sentiment_distribution_in_{language}_speaking_countries.png')

Contrary to my original assumption, there is no major difference in the sentiment composition between native speaker posts and non-native speaker posts. <br><br>However, some posts are labeled inappropriately based on our sentiment-nature analysis, which can lead to an overestimation of 'neutral' sentiments, <br>especially in non-English speaking countries.

## Correlation Analysis

Now we want to take a further step to analyze the correlation between some attributes thet we have explored in the EDA stage, and compute a numeric result.
<br><br>Especially in the following ways:
1. Find out how year, platform, and country impact sentiment scores of the social media posts. 
2. Find out which type of sentiment tend to get more reposts & likes. <br>Does positive posts tend to get more retweets and likes? <br>What about neutral posts and negative posts?

### Find out the correlation between 'Sentiment_Score' and 'Year', 'Platform', and 'Native Language'

In [ ]:
# create a new df for coorelation analysis
trim_data_1 = sentiment_data[['Sentiment_Nature', 'Platform', 'Native Language', 'Year']]
trim_data_1.head()

In [ ]:
# use pd.get_dummies to categorize columns
dummy_data_1 = pd.get_dummies(data=trim_data_1)
dummy_data_1.head()

In [ ]:
# run correlation analysis on the dummy_data
corr_matrix_1 = dummy_data_1.corr()
# see only the correlation between 'Sentiment_Nature' and other columns
pd.set_option('display.max_rows', None)
print("\033[1mPositive Sentiment Correlation:\033[0m")
print(f"{corr_matrix_1['Sentiment_Nature_positive'].sort_values(ascending=False)}\n")
print("\033[1mNeutral Sentiment Correlation:\033[0m")
print(f"{corr_matrix_1['Sentiment_Nature_neutral'].sort_values(ascending=False)}\n")
print("\033[1mNegative Sentiment Correlation:\033[0m")
print(f"{corr_matrix_1['Sentiment_Nature_negative'].sort_values(ascending=False)}")

In [ ]:
# visualize the correlation matrix using heatmap
plt.figure(figsize=(20,20))
sns.heatmap(corr_matrix_1, cmap='coolwarm', annot=True)
plt.savefig('correlation_on_sentiments_to_other_impact.png')

Based on the correlation analysis, it is interesting to see: 
1. *Positive* posts are more time-sensitive, and becoming more positive over the year.
2. *Neutral* posts are more language-sensitive, where **non-English speaker** tend to write neutral posts, which might be due to the language barrier, or mislabeling in sentiments.
3. On the other hand, *negative* posts are more language-sensitive, in terms of having **English speaker** writing negiva posts, which might be due to the tendency to express negative feelings in native language.

### Find out what kind of sentiment tend to get more reposts & likes

In [ ]:
# create a correlation matrix for sentiment natures and reposts & likes
trim_data_2 = sentiment_data[['Sentiment_Nature', 'Retweets', 'Likes']]
dummy_data_2 = pd.get_dummies(data=trim_data_2)
corr_matrix_2 = dummy_data_2.corr()

pd.set_option('display.max_rows', None)
print("\033[1mPositive Sentiment Correlation:\033[0m")
print(f"{corr_matrix_2['Sentiment_Nature_positive'].sort_values(ascending=False)}\n")
print("\033[1mNeutral Sentiment Correlation:\033[0m")
print(f"{corr_matrix_2['Sentiment_Nature_neutral'].sort_values(ascending=False)}\n")
print("\033[1mNegative Sentiment Correlation:\033[0m")
print(f"{corr_matrix_2['Sentiment_Nature_negative'].sort_values(ascending=False)}")

In [ ]:
sns.heatmap(corr_matrix_2, cmap='coolwarm', annot=True)
plt.savefig('correlation_on_sentiments_to_retweets_and_likes.png')

Based on the correlation analysis, it is interesting to see:
<br>
1. *Positive* posts tend to get more retweets and likes, which validate our hypothesis.
2. *Neutral* posts have tend to get less retweets and likes, but they are still positively correlated.
3. *Negative* posts shows all negative coorelation, which shows that negative posts do not tend to get more likes and retweets.

*Thank you for reading!*